# Windowed SBS Analysis — opportunity-adjusted

## Config

In [9]:
# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
import os, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.optimize import nnls
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform, cosine
from concurrent.futures import ProcessPoolExecutor, as_completed
from SigProfilerMatrixGenerator.scripts import SigProfilerMatrixGeneratorFunc as matGen

MUTATION_DIR = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/mutations_1kb/"
BED_1KB_DIR  = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/bedfiles_1kb/"
COSMIC_REF   = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/COSMIC_v3.5_SBS_GRCh38.txt"
GENOME_FASTA = "/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa"

OUT_ADJ    = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/"
OUT_SHARED = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_v2_25/"
TMP_DIR    = "/media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/tmp_windowed_v2_25/"
for d in [OUT_ADJ, OUT_SHARED, TMP_DIR]:
    os.makedirs(d, exist_ok=True)

WINDOW_SIZE        = 25
N_WINDOWS          = 40
ACTIVITY_THRESHOLD = 0.05
REFIT_AFTER_THRESHOLD = True

def window_pos(w):
    """Position of window w's centre, relative to the feature at the middle
    window (e.g. the TSS). Change WINDOW_SIZE/N_WINDOWS above only — every
    plot/CSV in this notebook calls this instead of hardcoding the arithmetic."""
    return (w + 1) * WINDOW_SIZE - (N_WINDOWS * WINDOW_SIZE) // 2

# Ref/alt in the mutation files are called relative to the GENE's sense
# strand, not the genome's plus strand — minus-strand genes need RC before
# the allele will match the genome. Leave True unless you've confirmed your
# mutation files already report genome-plus-strand alleles.
APPLY_REVCOMP = True

N_WORKERS = max(1, os.cpu_count() - 2)

# name: (mutation file, has_strand_column, n_cols, legacy_apply_rc_flag)
REGION_FILES = {
    "All_protein_coding_genes":                     (os.path.join(MUTATION_DIR, "Protein_coding_gene_rareSNP.txt"),  True, 14, True),
    "lncRNA_expressed_in_testes":                   (os.path.join(MUTATION_DIR, "lncRNA.txt"),                       True, 14, True),
    "Protein_coding_genes_not_expressed_in_testes": (os.path.join(MUTATION_DIR, "Non_testes_expressed_gene.txt"),    True, 14, True),
    "lncRNA_not_expressed_in_testes":               (os.path.join(MUTATION_DIR, "Non_testes_expressed_lncRNA.txt"),  True, 14, True),
    "Intergenic_RNAPII_pause_sites":                (os.path.join(MUTATION_DIR, "RNAPII.txt"),                       True, 14, True),
    "Random_intergenic":                            (os.path.join(MUTATION_DIR, "Intergenic_random.txt"),            True, 15, True),
}

PLOT_ORDER = [
    "All_protein_coding_genes",
    "lncRNA_expressed_in_testes",
    "Intergenic_RNAPII_pause_sites",
    "Protein_coding_genes_not_expressed_in_testes",
    "lncRNA_not_expressed_in_testes",
    "Random_intergenic",
]

# Pinned so it always draws last (= on top) and reads as a neutral catch-all
# rather than participating in the color gradient.
PINNED_TOP    = ["SBS5"]
PINNED_COLORS = {"SBS5": (0.55, 0.55, 0.55, 1.0)}

MIN_PCT_TO_LABEL = 8.0

cosmic      = pd.read_csv(COSMIC_REF, sep="\t", index_col=0)
sigs_to_fit = cosmic.columns.tolist()
channels    = cosmic.index.tolist()
cosmic_mat  = cosmic.loc[channels].values

def opportunity_ratio(opp_local, opp_genome):
    """Channel-wise (genome fraction) / (local fraction), mean-normalised to 1
    so fitted activities stay on the scale of the observed mutation count.
    Defined here (not in Step 5) so D1/D2 diagnostics work even if you skip
    straight to the Step 5 RELOAD cell instead of running Step 5 itself."""
    n = opp_local.reindex(channels, fill_value=0).values.astype(float)
    N = opp_genome.reindex(channels).values.astype(float)
    r = np.divide(N / N.sum(), n / n.sum(), out=np.ones_like(N), where=n > 0)
    return r / r.mean()

print(f"Regions: {list(REGION_FILES)}")
print(f"Windows: {N_WINDOWS} x {WINDOW_SIZE} bp | Workers: {N_WORKERS}")
print(f"COSMIC: {len(sigs_to_fit)} signatures x {len(channels)} channels")


Regions: ['All_protein_coding_genes', 'lncRNA_expressed_in_testes', 'Protein_coding_genes_not_expressed_in_testes', 'lncRNA_not_expressed_in_testes', 'Intergenic_RNAPII_pause_sites', 'Random_intergenic']
Windows: 40 x 25 bp | Workers: 30
COSMIC: 97 signatures x 96 channels


## Steps 1–3: Parse mutations → write VCFs → build SBS96 matrices
*(Skip and use the Reload cell if already computed)*

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 1: Parse mutation files -> assign each mutation to a window
# ─────────────────────────────────────────────────────────────

def parse_mutation_file(name, path, has_strand, ncols, apply_rc):
    rows = []
    with open(path) as f:
        for line in f:
            cols = line.strip().split()
            if len(cols) < 4:
                continue
            try:
                rel_pos = int(cols[0])
                mut     = cols[1]
                if ">" not in mut:
                    continue
                parts = mut.split(">")
                ref   = parts[0][-1]
                alt   = parts[1][0]
                if ref not in "ACGT" or alt not in "ACGT":
                    continue
                strand = cols[-1] if has_strand else "+"
                pos    = int(cols[-2])
                chrom  = str(cols[-3])
                window = min(rel_pos // WINDOW_SIZE, N_WINDOWS - 1)
                rows.append({"rel_pos": rel_pos, "window": window, "ref": ref,
                             "alt": alt, "chrom": chrom, "pos": pos,
                             "strand": strand})
            except (ValueError, IndexError):
                continue

    if not rows:
        print(f"  WARNING: {name} — no valid mutations parsed")
        return pd.DataFrame(columns=["rel_pos","window","ref","alt","chrom","pos","strand"])

    df = pd.DataFrame(rows)
    n_minus = (df["strand"] == "-").sum()
    print(f"  {name}: {len(df):,} mutations | {n_minus:,} on minus strand "
          f"({100*n_minus/len(df):.1f}%) | {df['window'].nunique()} windows")
    return df

all_mutations = {}
for name, (path, has_strand, ncols, apply_rc) in REGION_FILES.items():
    all_mutations[name] = parse_mutation_file(name, path, has_strand, ncols, apply_rc)

print("\nMutation loading complete.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 2: Write one VCF per (region, window)
# ─────────────────────────────────────────────────────────────
COMPLEMENT = str.maketrans("ACGT", "TGCA")

def mut_to_vcf_row(chrom, pos, ref, alt, strand):
    if APPLY_REVCOMP and strand == "-":
        ref = ref.translate(COMPLEMENT)
        alt = alt.translate(COMPLEMENT)
    return f"{chrom.replace('chr','')}\t{pos}\t.\t{ref}\t{alt}\t.\t.\t.\n"

def write_window_vcf_task(args):
    name, w, records, vcf_dir = args
    os.makedirs(vcf_dir, exist_ok=True)
    path = os.path.join(vcf_dir, f"{name}_w{w:02d}.vcf")
    n_written, seen = 0, set()
    with open(path, "w") as f:
        f.write("##fileformat=VCFv4.1\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        for r in records:
            key = (r["chrom"], r["pos"], r["ref"], r["alt"])
            if key in seen:
                continue
            seen.add(key)
            f.write(mut_to_vcf_row(r["chrom"], r["pos"], r["ref"], r["alt"], r["strand"]))
            n_written += 1
    return (name, w, vcf_dir, n_written)

tasks = []
for name, df in all_mutations.items():
    for w in range(N_WINDOWS):
        df_w = df[df["window"] == w]
        if len(df_w) == 0:
            continue
        vcf_dir = os.path.join(TMP_DIR, f"{name}_w{w:02d}")
        tasks.append((name, w, df_w.to_dict("records"), vcf_dir))

print(f"Writing {len(tasks)} VCF files with {N_WORKERS} workers...")
window_vcf_dirs    = {name: {} for name in REGION_FILES}
vcf_written_counts = {name: {} for name in REGION_FILES}

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    for fut in as_completed([ex.submit(write_window_vcf_task, t) for t in tasks]):
        name, w, vcf_dir, n = fut.result()
        window_vcf_dirs[name][w]    = vcf_dir
        vcf_written_counts[name][w] = n

print(f"Done. {sum(len(v) for v in window_vcf_dirs.values())} VCFs written.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 3: SBS96 mutation matrices, one per (region, window)
# ─────────────────────────────────────────────────────────────

def generate_matrix_for_window(args):
    name, w, vcf_dir = args
    try:
        matrices = matGen.SigProfilerMatrixGeneratorFunc(
            f"{name}_w{w:02d}", "GRCh38", vcf_dir,
            exome=False, bed_file=None, chrom_based=False,
            plot=False, seqInfo=False,
        )
        if "96" in matrices:
            return (name, w, matrices["96"].sum(axis=1), None)
        return (name, w, None, "no 96 matrix returned")
    except Exception as e:
        return (name, w, None, repr(e))

tasks = [(name, w, d) for name, ws in window_vcf_dirs.items() for w, d in ws.items()]
print(f"Running SigProfilerMatrixGenerator on {len(tasks)} windows...")

window_mut_matrices = {name: {} for name in REGION_FILES}
failures = []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    for fut in as_completed([ex.submit(generate_matrix_for_window, t) for t in tasks]):
        name, w, result, err = fut.result()
        if result is not None:
            window_mut_matrices[name][w] = result
        else:
            failures.append((name, w, err))

print("\nMatrix generation complete.")
for name in REGION_FILES:
    print(f"  {name}: {len(window_mut_matrices[name])}/{N_WINDOWS} windows")
if failures:
    print("\nFAILURES:")
    for f in failures[:10]:
        print("  ", f)

# ── QC: how many written variants actually survived the reference check? ──
# A region-wide collapse to ~0% almost always means APPLY_REVCOMP is set
# wrong for that region's strand convention — check that before anything else.
print("\nRetention (matrix total / VCF lines written) — should be ~100%:")
for name in REGION_FILES:
    got  = sum(window_mut_matrices[name][w].sum() for w in window_mut_matrices[name])
    want = sum(vcf_written_counts[name].values())
    pct  = 100 * got / max(want, 1)
    flag = "  <-- check APPLY_REVCOMP for this region" if pct < 90 else ""
    print(f"  {name:48s} {got:>10,.0f} / {want:>10,.0f}  = {pct:5.1f}%{flag}")

with open(os.path.join(OUT_SHARED, "window_mut_matrices.pkl"), "wb") as f:
    pickle.dump(window_mut_matrices, f)


## Reload: skip Steps 1–3 (after kernel restart)

In [10]:
# ─────────────────────────────────────────────────────────────
# RELOAD: mutation matrices
# ─────────────────────────────────────────────────────────────
p = os.path.join(OUT_SHARED, "window_mut_matrices.pkl")
if os.path.exists(p):
    with open(p, "rb") as f:
        window_mut_matrices = pickle.load(f)
    print(f"Loaded: {p}")
    for name in REGION_FILES:
        n = sum(window_mut_matrices.get(name, {}).get(w, pd.Series(dtype=float)).sum()
                for w in range(N_WINDOWS))
        print(f"  {name:48s} {len(window_mut_matrices.get(name, {})):2d} windows, {n:,.0f} mutations")
else:
    print("Not found — run Steps 1-3.")


Loaded: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_v2_25/window_mut_matrices.pkl
  All_protein_coding_genes                         40 windows, 3,866,948 mutations
  lncRNA_expressed_in_testes                       40 windows, 936,436 mutations
  Protein_coding_genes_not_expressed_in_testes     40 windows, 160,702 mutations
  lncRNA_not_expressed_in_testes                   40 windows, 2,323,166 mutations
  Intergenic_RNAPII_pause_sites                    40 windows, 276,172 mutations
  Random_intergenic                                40 windows, 2,417,493 mutations


## Step 4: trinucleotide opportunities

`n_i` = number of sites in each window whose reference trinucleotide
corresponds to channel *i*; `N_i` = the same count over the whole reference
genome — both counted directly from the FASTA. Counts are pyrimidine-collapsed
to match the SBS96 convention.

*(Slow — a few minutes. Skip and use the Reload cell if already computed.)*

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 4: count trinucleotide SITES (not mutations) from the FASTA
# ─────────────────────────────────────────────────────────────
from pyfaidx import Fasta

_CODE = np.full(256, 4, dtype=np.uint8)
for _i, _b in enumerate(b"ACGT"):
    _CODE[_b] = _i
    _CODE[_b + 32] = _i

_BASES  = "ACGT"
_TRINUC = [a + b + c for a in _BASES for b in _BASES for c in _BASES]
_COMP   = str.maketrans("ACGT", "TGCA")
_rc     = lambda s: s.translate(_COMP)[::-1]

def count_trinucs(seq_blocks):
    """Pyrimidine-collapsed trinucleotide counts over a list of sequences.
    Each block carries 1 bp of flank on both sides, so a block of length L+2
    contributes L trinucleotides — one centred on every analysed site."""
    arr  = np.frombuffer("N".join(seq_blocks).encode(), dtype=np.uint8)
    c    = _CODE[arr].astype(np.int32)
    hist = np.bincount(c[:-2] * 25 + c[1:-1] * 5 + c[2:], minlength=125)
    out = {}
    for t in _TRINUC:
        i = _BASES.index(t[0]) * 25 + _BASES.index(t[1]) * 5 + _BASES.index(t[2])
        k = t if t[1] in "CT" else _rc(t)
        out[k] = out.get(k, 0) + int(hist[i])
    return out

def to_channels(counts, channels):
    return pd.Series([float(counts.get(ch[0] + ch[2] + ch[6], 0)) for ch in channels],
                     index=channels)

def fetch(fa, chrom, start, end):
    key = chrom if chrom in fa else chrom.replace("chr", "")
    if key not in fa:
        return None
    L = len(fa[key])
    s, e = max(0, start - 1), min(L, end + 1)
    return str(fa[key][s:e]).upper() if e - s >= 3 else None

fa = Fasta(GENOME_FASTA, sequence_always_upper=True)

def load_bed(bed_path):
    STANDARD = set([f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"])
    out = []
    with open(bed_path) as f:
        for line in f:
            col = line.split()
            if len(col) >= 3 and col[0] in STANDARD:
                out.append((col[0], int(col[1]), int(col[2]),
                            col[5] if len(col) > 5 else "+"))
    return out

# ── local opportunities: one 96-vector per (region, window) ────────────────
all_window_opp = {}
for name in REGION_FILES:
    bed_path = os.path.join(BED_1KB_DIR, name + ".bed")
    if not os.path.exists(bed_path):
        print(f"  WARNING: no BED for {name}")
        continue
    regions = load_bed(bed_path)
    all_window_opp[name] = {}
    for w in range(N_WINDOWS):
        ws, we = w * WINDOW_SIZE, (w + 1) * WINDOW_SIZE
        blocks = []
        for chrom, start, end, strand in regions:
            if strand == "+":
                gs, ge = start + ws, start + we
            else:
                gs, ge = end - we, end - ws
            seq = fetch(fa, chrom, max(0, gs), ge)
            if seq:
                blocks.append(seq)
        all_window_opp[name][w] = to_channels(count_trinucs(blocks), channels)
    n0 = all_window_opp[name][0].sum() / 3
    print(f"  {name:48s} {len(regions):>7,} regions | window 0 = {n0:>12,.0f} sites")

# ── genome-wide opportunities (the COSMIC denominator) ──────────────────────
print("\nCounting genome-wide trinucleotides...")
CHUNK = 20_000_000
genome_counts = {}
for c in [str(i) for i in range(1, 23)] + ["X", "Y"]:
    key = c if c in fa else "chr" + c
    if key not in fa:
        print(f"  skipping {c} (not in FASTA)")
        continue
    L = len(fa[key])
    for s in range(0, L, CHUNK):
        e   = min(L, s + CHUNK)
        seq = str(fa[key][max(0, s - 1):min(L, e + 1)]).upper()
        for k, v in count_trinucs([seq]).items():
            genome_counts[k] = genome_counts.get(k, 0) + v
    print(f"  chr{c}: {L:,} bp")

genome_opp = to_channels(genome_counts, channels)
print(f"\nGenome-wide analysable sites: {genome_opp.sum()/3:,.0f}")

with open(os.path.join(OUT_SHARED, "opportunities.pkl"), "wb") as f:
    pickle.dump({"local": all_window_opp, "genome": genome_opp}, f)
print("Saved opportunities.pkl")


## Reload: skip Step 4 (after kernel restart)

In [11]:
# ─────────────────────────────────────────────────────────────
# RELOAD: opportunities
# ─────────────────────────────────────────────────────────────
p = os.path.join(OUT_SHARED, "opportunities.pkl")
if os.path.exists(p):
    with open(p, "rb") as f:
        _opp = pickle.load(f)
    all_window_opp, genome_opp = _opp["local"], _opp["genome"]
    print(f"Loaded: {p}")
    print(f"  genome-wide sites: {genome_opp.sum()/3:,.0f}")
    for name in all_window_opp:
        print(f"  {name:48s} {len(all_window_opp[name])} windows")
else:
    print("Not found — run Step 4.")


Loaded: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_v2_25/opportunities.pkl
  genome-wide sites: 2,934,858,504
  All_protein_coding_genes                         40 windows
  lncRNA_expressed_in_testes                       40 windows
  Protein_coding_genes_not_expressed_in_testes     40 windows
  lncRNA_not_expressed_in_testes                   40 windows
  Intergenic_RNAPII_pause_sites                    40 windows
  Random_intergenic                                40 windows


## Step 5: opportunity-adjusted fitting

COSMIC probabilities carry the genome's composition already: `p_ki ∝ mu_ki *
N_i`. In a window the expected spectrum is `sum_k a_k * p_ki * (n_i / N_i)`,
so the observed counts are rescaled by `N_i / n_i` (each expressed as a
fraction of its own total) and then fitted against the unmodified COSMIC
matrix.

In [ ]:
# ─────────────────────────────────────────────────────────────
# STEP 5: opportunity-adjusted NNLS fit
# ─────────────────────────────────────────────────────────────

def _nnls_with_threshold(A, y, threshold):
    a = nnls(A, y)[0]
    if a.sum() <= 0:
        return pd.Series(0.0, index=sigs_to_fit)
    keep = a >= threshold * a.sum()
    if REFIT_AFTER_THRESHOLD and keep.any() and not keep.all():
        a_keep = nnls(A[:, keep], y)[0]
    else:
        a_keep = a[keep]
    out = pd.Series(0.0, index=sigs_to_fit)
    out.iloc[np.flatnonzero(keep)] = a_keep
    return out

def fit_adjusted(observed, opp_local, opp_genome, threshold=ACTIVITY_THRESHOLD):
    o = observed.reindex(channels, fill_value=0).values.astype(float)
    return _nnls_with_threshold(cosmic_mat, o * opportunity_ratio(opp_local, opp_genome),
                                threshold)

all_window_activities_adj = {}
for name in REGION_FILES:
    acts = {}
    for w in range(N_WINDOWS):
        if w not in window_mut_matrices.get(name, {}):
            acts[w] = pd.Series(0.0, index=sigs_to_fit)
            continue
        acts[w] = fit_adjusted(window_mut_matrices[name][w],
                               all_window_opp[name][w], genome_opp)
    df = pd.DataFrame(acts).T
    df.index.name = "window"
    all_window_activities_adj[name] = df
    print(f"  {name:48s} {df.columns[(df > 0).any(axis=0)].tolist()}")

with open(os.path.join(OUT_ADJ, "activities_adjusted.pkl"), "wb") as f:
    pickle.dump(all_window_activities_adj, f)
print(f"Saved: {os.path.join(OUT_ADJ, 'activities_adjusted.pkl')}")


## Reload: skip Step 5 (after kernel restart)

In [12]:
# ─────────────────────────────────────────────────────────────
# RELOAD: fitted activities
# ─────────────────────────────────────────────────────────────
p = os.path.join(OUT_ADJ, "activities_adjusted.pkl")
if os.path.exists(p):
    with open(p, "rb") as f:
        all_window_activities_adj = pickle.load(f)
    print(f"Loaded: {p}")
else:
    print("Not found — run Step 5.")


Loaded: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/activities_adjusted.pkl


## Signature order and colors

Order is a similarity clustering of only the signatures this fit actually
uses; colors are a smooth hue sweep blended toward white for a pastel look, with SBS5
pinned to a fixed neutral grey and always drawn last.

In [36]:
# ─────────────────────────────────────────────────────────────
# Order + colors, computed fresh from all_window_activities_adj
# ─────────────────────────────────────────────────────────────
def tint_toward_white(rgb, amount=0.35):
    r, g, b = rgb
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount, 1.0)

sigs_adj = sorted(set(
    s for df in all_window_activities_adj.values()
    for s in df.columns[(df > 0).any(axis=0)]
))
print(f"Active signatures ({len(sigs_adj)}): {sigs_adj}")

cluster_sigs = [s for s in sigs_adj if s not in PINNED_COLORS]
cosmic_sub   = cosmic[[s for s in cluster_sigs if s in cosmic.columns]]
n = len(cluster_sigs)
sim = pd.DataFrame(np.zeros((n, n)), index=cluster_sigs, columns=cluster_sigs)
for s1 in cluster_sigs:
    for s2 in cluster_sigs:
        sim.loc[s1, s2] = 1 - cosine(cosmic_sub[s1].values, cosmic_sub[s2].values)

order_idx = leaves_list(linkage(squareform(1 - sim.values, checks=False), method="average"))
adj_order = [cluster_sigs[i] for i in order_idx] + [s for s in PINNED_TOP if s in sigs_adj]

gradient_sigs = [s for s in adj_order if s not in PINNED_COLORS]
hues = np.linspace(0.02, 0.90, len(gradient_sigs))
sig_colors = {sig: tint_toward_white(mcolors.hsv_to_rgb([h, 0.75, 0.95]))
             for sig, h in zip(gradient_sigs, hues)}
sig_colors.update(PINNED_COLORS)

print(f"Order ({len(adj_order)}): {adj_order}")


Active signatures (23): ['SBS110', 'SBS12', 'SBS16', 'SBS19', 'SBS26', 'SBS3', 'SBS30', 'SBS31', 'SBS32', 'SBS37', 'SBS39', 'SBS40b', 'SBS40c', 'SBS5', 'SBS54', 'SBS57', 'SBS58', 'SBS84', 'SBS88', 'SBS89', 'SBS92', 'SBS93', 'SBS96']
Order (23): ['SBS19', 'SBS31', 'SBS84', 'SBS32', 'SBS110', 'SBS30', 'SBS54', 'SBS96', 'SBS40b', 'SBS93', 'SBS40c', 'SBS92', 'SBS39', 'SBS3', 'SBS89', 'SBS37', 'SBS12', 'SBS26', 'SBS16', 'SBS88', 'SBS57', 'SBS58', 'SBS5']


## Stacked-bar plot

In [31]:
# ─────────────────────────────────────────────────────────────
# Stacked-bar figure — opportunity-adjusted fit
# ─────────────────────────────────────────────────────────────
MIN_PCT_TO_LABEL = 1.0

def to_percent(df):
    out = df.div(df.sum(axis=1).replace(0, np.nan), axis=0) * 100
    return out.fillna(0)

fig, axes = plt.subplots(2, 3, figsize=(33, 10))
axes = axes.flatten()
for idx, name in enumerate(PLOT_ORDER):
    ax = axes[idx]
    df_pct = to_percent(all_window_activities_adj[name])
    bottom = np.zeros(N_WINDOWS)
    #label_fontsize = max(5, min(8, 8 - (N_WINDOWS - 20) // 10))
    label_fontsize = 4
    for sig in [s for s in adj_order if s in df_pct.columns and df_pct[s].sum() > 0]:
        vals = df_pct[sig].reindex(range(N_WINDOWS), fill_value=0).values
        ax.bar(range(N_WINDOWS), vals, bottom=bottom,
               color=sig_colors[sig], width=0.85, edgecolor="none")
        for w in range(N_WINDOWS):
            if vals[w] >= MIN_PCT_TO_LABEL:
                ax.text(w, bottom[w] + vals[w] / 2, sig, ha="center", va="center",
                        fontsize=label_fontsize, clip_on=True)
        bottom += vals
    ax.set_title(name.replace("_", " "), fontsize=18, fontweight="bold")
    ax.set_xlabel("Position relative to TSS/focal site", fontsize=18)
    ax.set_ylabel("% Mutations", fontsize=18)
    ax.tick_params(axis="y", labelsize=14)
    ax.set_xticks([w + 0.5 for w in range(N_WINDOWS)])
    LABEL_STEP = max(1, round(N_WINDOWS / 20))
    tick_fontsize = max(8, min(14, 14 - (N_WINDOWS - 20) // 10))
    labels = [f"{window_pos(w):+d}".replace("+0", "0") if w % LABEL_STEP == 0 else ""
             for w in range(N_WINDOWS)]
    ax.set_xticklabels(labels, rotation=90, fontsize=tick_fontsize)
    ax.set_ylim(0, 100)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
for idx in range(len(PLOT_ORDER), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("SBS Signature Activities Across Windows\n(opportunity-adjusted: local -> genome-wide)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
out = os.path.join(OUT_ADJ, "sbs_windowed_stacked_adjusted.svg")
plt.savefig(out, format="svg", bbox_inches="tight"); print(f"Saved: {out}")
plt.show()

# CSV
frames = []
for name in PLOT_ORDER:
    d = to_percent(all_window_activities_adj[name])
    d.index = [f"{window_pos(w):+d}" for w in range(N_WINDOWS)]
    d.index.name = "position_relative_to_TSS"
    d.insert(0, "region", name)
    frames.append(d)
allf = pd.concat(frames)
cols = [s for s in adj_order if s in allf.columns]
allf = allf[["region"] + cols]
csv_out = os.path.join(OUT_ADJ, "all_regions_sbs_activities_adjusted.csv")
allf.to_csv(csv_out); print(f"Saved: {csv_out}")


Saved: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/sbs_windowed_stacked_adjusted.svg
Saved: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/all_regions_sbs_activities_adjusted.csv


## Legend

In [17]:
# ─────────────────────────────────────────────────────────────
# LEGEND: opportunity-adjusted signatures
# ─────────────────────────────────────────────────────────────
ncols_leg = 5
fig_leg, ax_leg = plt.subplots(figsize=(ncols_leg * 3, int(np.ceil(len(adj_order)/ncols_leg)) * 0.6 + 0.5))
ax_leg.set_axis_off()
ax_leg.legend(
    handles=[mpatches.Patch(color=sig_colors[s], label=s) for s in adj_order],
    title="SBS Signature (opportunity-adjusted, ordered by similarity)",
    loc="center", ncol=ncols_leg, fontsize=10, title_fontsize=11,
    frameon=True, handlelength=2, handleheight=1.2,
)
plt.tight_layout()
leg_out = os.path.join(OUT_ADJ, "sbs_adjusted_legend.svg")
plt.savefig(leg_out, format="svg", bbox_inches="tight")
print(f"Legend saved: {leg_out}  ({len(adj_order)} signatures)")
plt.show()


Legend saved: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/sbs_adjusted_legend.svg  (23 signatures)


## Legend table with etiology

In [33]:
# ─────────────────────────────────────────────────────────────
# LEGEND TABLE: signature -> proposed etiology
# ─────────────────────────────────────────────────────────────
etiology = {
    "SBS98":  "Similar to SBS87, C>G dominant variant",
    "SBS15":  "Defective DNA mismatch repair",
    "SBS6":   "Defective DNA mismatch repair",
    "SBS1":   "Spontaneous deamination of 5mC at CpG",
    "SBS87":  "Thiopurine chemotherapy treatment",
    "SBS24":  "Aflatoxin exposure",
    "SBS31":  "Platinum chemotherapy treatment",
    "SBS19":  "Unknown, C>T",
    "SBS23":  "Unknown, C>T",
    "SBS11":  "Temozolomide treatment, C>T",
    "SBS32":  "Azathioprine treatment, C>T",
    "SBS7b":  "Ultraviolet light exposure, C>T",
    "SBS102": "Unknown, C>T",
    "SBS30":  "Defective base excision repair",
    "SBS39":  "HR deficiency, dsDNA break repair",
    "SBS86":  "Unknown chemotherapy treatment",
    "SBS57":  "Possible sequencing artefact",
    "SBS16":  "Strong transcription strand bias",
    "SBS3":   "Defective HR DNA damage repair",
    "SBS40c": "Transcription-coupled mutagenesis,\nnon-canonical DSB repair",
    "SBS40b":  "Unknown",
    "SBS5":   "Unknown; clock-like",
    "SBS54":  "Possible sequencing artefact",
    "SBS46":  "Possible sequencing artefact",
    "SBS12":  "Transcription strand bias",
    "SBS26":  "Defective DNA mismatch repair",
    "SBS84":  "Activation-induced cytidine deaminase",
    "SBS110":  "Unknown",
    "SBS96":  "Unknown",
    "SBS93":  "Unknown",
    "SBS92":  "Tobacco smoking",
    "SBS89":  "Unknown",
    "SBS37":  "Unknown",
    "SBS88":  "Colibactin exposure",
    "SBS58":  "Sequencing artefact",
}

n = len(adj_order)
row_height = 0.35
fig_height = n * row_height + 0.8

fig, ax = plt.subplots(figsize=(6, fig_height))
ax.set_xlim(0, 6)
ax.set_ylim(0, fig_height)
ax.set_axis_off()

ax.text(0.3,  fig_height - 0.3, "SBS", fontsize=10, fontweight="bold", va="top")
ax.text(1.2,  fig_height - 0.3, "Signature", fontsize=10, fontweight="bold", va="top")
ax.text(2.6,  fig_height - 0.3, "Proposed Etiology", fontsize=10, fontweight="bold", va="top")
ax.axhline(y=fig_height - 0.55, xmin=0.0, xmax=1.0, color="black", linewidth=0.8)

for i, sig in enumerate(adj_order):
    y = fig_height - 0.7 - i * row_height
    color = sig_colors.get(sig, (0.9, 0.9, 0.9, 1.0))

    rect = mpatches.FancyBboxPatch((0.05, y - 0.1), 0.7, 0.25,
                                    boxstyle="round,pad=0.02",
                                    facecolor=color, edgecolor="none")
    ax.add_patch(rect)

    ax.text(0.9, y + 0.07, sig, fontsize=8.5, va="center", fontweight="bold")
    ax.text(2.6, y + 0.07, etiology.get(sig, ""), fontsize=8, va="center")

    if i < n - 1:
        ax.axhline(y=y - 0.14, xmin=0.0, xmax=1.0, color="lightgrey", linewidth=0.4)

plt.tight_layout(pad=0.3)
leg_out = os.path.join(OUT_ADJ, "sbs_adjusted_legend_table.svg")
plt.savefig(leg_out, format="svg", bbox_inches="tight")
print(f"Saved: {leg_out}")
plt.show()


Saved: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/sbs_adjusted_legend_table.svg


## Pairwise cosine similarity

In [39]:
# ─────────────────────────────────────────────────────────────
# PAIRWISE COSINE SIMILARITY: opportunity-adjusted signatures
# Insert SBS5 next to its nearest neighbor within the ACTUAL clustered
# gradient order (adj_order, minus the pinned SBS5 at its end) — not
# cluster_sigs, which is just an alphabetically-sorted pre-clustering list.
# ─────────────────────────────────────────────────────────────
gradient_order = [s for s in adj_order if s not in PINNED_COLORS]

target = "SBS5"
if target in sigs_adj and target not in gradient_order:
    sims_to_target = {s: 1 - cosine(cosmic[s].values, cosmic[target].values)
                      for s in gradient_order if s in cosmic.columns}
    best_neighbor = max(sims_to_target, key=sims_to_target.get)
    insert_pos = gradient_order.index(best_neighbor) + 1
    cosine_order = gradient_order[:insert_pos] + [target] + gradient_order[insert_pos:]
    print(f"SBS5 inserted next to its closest neighbor: {best_neighbor} "
         f"(cosine similarity {sims_to_target[best_neighbor]:.3f})")
else:
    cosine_order = gradient_order

cosmic_adj = cosmic[[s for s in cosine_order if s in cosmic.columns]]
n = len(cosine_order)
pairwise_sim_adj = pd.DataFrame(np.zeros((n, n)), index=cosine_order, columns=cosine_order)
for s1 in cosine_order:
    for s2 in cosine_order:
        pairwise_sim_adj.loc[s1, s2] = 1 - cosine(cosmic_adj[s1].values, cosmic_adj[s2].values)
pairwise_sim_adj = pairwise_sim_adj.astype(float)

fig, ax = plt.subplots(figsize=(max(8, n * 0.65), max(7, n * 0.65)))
sns.heatmap(pairwise_sim_adj, ax=ax, cmap="GnBu", vmin=0, vmax=1,
            linewidths=0.5, linecolor="lightgrey", annot=True, fmt=".2f",
            square=True, cbar_kws={"label": "Cosine Similarity", "shrink": 0.6},
            xticklabels=False, yticklabels=False)
ax.xaxis.set_ticks_position('bottom')
ax.xaxis.set_label_position('bottom')
ax.set_title("Pairwise Cosine Similarity Between COSMIC SBS Reference Profiles\n(opportunity-adjusted signatures only)",
             fontsize=13, fontweight="bold", pad=40)

for i, sig in enumerate(cosine_order):
    color = sig_colors.get(sig, (0.9, 0.9, 0.9, 1.0))
    ax.text(i + 0.5, n + 0.3, sig, ha="center", va="top", fontsize=14, fontweight="bold",
            rotation=90, bbox=dict(boxstyle="square,pad=0.6", facecolor=color, edgecolor="none"))
    ax.text(-0.3, i + 0.5, sig, ha="right", va="center", fontsize=14, fontweight="bold",
            bbox=dict(boxstyle="square,pad=0.6", facecolor=color, edgecolor="none"))

ax.tick_params(axis="both", length=0)
plt.tight_layout()
out = os.path.join(OUT_ADJ, "sbs_adjusted_pairwise_cosine.svg")
plt.savefig(out, format="svg", bbox_inches="tight")
print(f"Saved: {out}")
plt.show()
pairwise_sim_adj.to_csv(out.replace(".svg", ".csv"))

SBS5 inserted next to its closest neighbor: SBS40c (cosine similarity 0.911)
Saved: /media/alexpalazzo1/ohta/Tina/TSS_hypermutation/COSMIC/output_windowed_opp_adjusted_25/sbs_adjusted_pairwise_cosine.svg
